# KIS: tạo artifacts từ keyframes có sẵn trên Kaggle

Thay cho `videoprocessing.ipynb`: **không cần video, không giải mã video, không trích xuất/lấy mẫu lại keyframes**.

Giữ các bước: **SigLIP visual features → objects → InternVL caption → caption mapping + FAISS text index**. Model/revision và schema dùng lại pipeline hiện tại.

## Input cần có
- Ảnh JPEG, ví dụ `keyframes/L03/L03_V001/001.jpg` hoặc `L03_V001/001.jpg` (folder gốc có thể lồng nhiều cấp).
- Bỏ qua bản sao có hậu tố `(1)`, `(2)`… như `L21_V001/171 (1).jpg` hoặc `L21_V001 171 (1).jpg`; chỉ dùng ảnh gốc, không xóa file Input hay đổi mapping. Log ghi số bản sao đã bỏ qua. Nếu thiếu ảnh gốc trong CSV, vẫn báo lỗi coverage.
- CSV gốc của từng video, ví dụ `mapping/L03/L03_V001.csv`, với cột `n,pts_time,fps,frame_idx`.
- Hỗ trợ mapping L21 có nhiều keyframe cùng `frame_idx`: giữ cả ảnh và CSV nguyên vẹn, định danh bằng `(video_id, keyframe_n)`. `frame_idx` phải không giảm; alias được cảnh báo và lưu trong checkpoint/manifest, không coi là frame ID đã hiệu chỉnh.
- `001.jpg` là **keyframe number**, không tự coi là `frame_idx`. Nếu chưa có CSV, cần bổ sung mapping gốc; notebook sẽ không đoán FPS/timestamp. Không tự renumber khi thiếu ảnh.
- Nếu Input là ZIP chưa giải nén, giải nén vào folder riêng trong `/kaggle/working` rồi trỏ các ROOT vào folder đó. Không trỏ output vào folder input.
- Add Input gói code **mới** `kis-kaggle-from-keyframes.zip` đã giải nén, có `tools/kaggle_prepare_keyframes.py`. Gói cũ có thể chưa có runner này.

Bật **GPU** và **Internet** trong Kaggle. Chỉ sửa đường dẫn/config ở cell đầu, sau đó Run All.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Đổi theo mount path thực tế trong Kaggle Input.
CODE_DIR = Path('/kaggle/input/datasets/nghvitkhnh/keyframes-code/kis_kaggle')
KEYFRAMES_ROOT = Path('/kaggle/input/YOUR-KEYFRAMES-DATASET')
MAPPING_ROOT = Path('/kaggle/input/YOUR-MAPPING-DATASET')
OUTPUT_ROOT = Path('/kaggle/working/kis_from_keyframes')

# None: tất cả video tìm thấy; hoặc ['L03_V001', 'L03_V002'].
# Chạy thử một video bằng danh sách này, không cắt lại ảnh/timestamp.
VIDEO_IDS = None
VISUAL_BATCH = 8
OBJECT_BATCH = 8
CAPTION_BATCH = 16
CAPTION_BACKEND = 'internvl'
CAPTION_4BIT = False
RESTART_CAPTIONS = False  # Chỉ bật khi cố ý archive và tạo lại caption.
INCLUDE_IMAGES_IN_ZIP = False  # False: không tải lại ảnh đã có trong Input.

for folder in (CODE_DIR, KEYFRAMES_ROOT, MAPPING_ROOT):
    assert folder.is_dir(), f'Sửa đường dẫn: {folder}'
assert (CODE_DIR / 'tools/kaggle_prepare_keyframes.py').is_file(), 'Cần upload gói code mới kis-kaggle-from-keyframes.zip'
assert 'ignored_duplicate_files' in (CODE_DIR / 'tools/kaggle_prepare_keyframes.py').read_text(encoding='utf-8'), 'Cập nhật gói code mới để bỏ qua keyframe dạng 171 (1).jpg'
assert 'allow_frame_idx_aliases' in (CODE_DIR / 'tools/prepare_video_data.py').read_text(encoding='utf-8') and 'frame_idx_aliases' in (CODE_DIR / 'tools/kaggle_prepare_keyframes.py').read_text(encoding='utf-8'), 'Cần cập nhật toàn bộ gói code ZIP hỗ trợ mapping L21, không chỉ notebook'
assert (CODE_DIR / 'requirements-kaggle.txt').is_file()
assert OUTPUT_ROOT.resolve().is_relative_to(Path('/kaggle/working')), 'Output phải nằm trong /kaggle/working'
print('Input ảnh:', KEYFRAMES_ROOT)
print('Input mapping:', MAPPING_ROOT)
print('Output:', OUTPUT_ROOT)


In [ ]:
# Giữ CUDA torch/torchvision của Kaggle; cài dependencies pipeline full.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(CODE_DIR / 'requirements-kaggle.txt')], check=True)
ENV = dict(os.environ)
ENV['PYTHONPATH'] = str(CODE_DIR) + os.pathsep + ENV.get('PYTHONPATH', '')
ENV['TOKENIZERS_PARALLELISM'] = 'false'
# Tránh giới hạn nhóm từ một cấu hình môi trường khác.
ENV.pop('KIS_VIDEO_GROUPS', None)
print('Setup hoàn tất.')


In [ ]:
command = [
    sys.executable, '-u', '-m', 'tools.kaggle_prepare_keyframes',
    '--keyframes-root', str(KEYFRAMES_ROOT),
    '--mapping-root', str(MAPPING_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--visual-batch', str(VISUAL_BATCH),
    '--object-batch', str(OBJECT_BATCH),
    '--caption-batch', str(CAPTION_BATCH),
    '--caption-backend', CAPTION_BACKEND,
]
for video_id in VIDEO_IDS or []:
    command += ['--video-id', video_id]
if CAPTION_4BIT:
    command.append('--caption-4bit')
if RESTART_CAPTIONS:
    command.append('--restart-captions')
if INCLUDE_IMAGES_IN_ZIP:
    command.append('--include-images')

# Preflight toàn bộ ảnh, hash và mapping; chưa nạp model, chưa tạo output.
subprocess.run(command + ['--check-only'], cwd=CODE_DIR, env=ENV, check=True)


## Chạy pipeline từ ảnh

Ảnh và mapping được copy nguyên bytes sang Working, kiểm hash và đăng ký checkpoint. Các model chạy lần lượt để giải phóng VRAM giữa các stage. **Không gọi bước extract keyframes.**

Resume bằng cùng output/config; checkpoint khác input bị chặn thay vì ghi đè. Sau khi đổi Kaggle session, khôi phục toàn bộ output Working cũ (bao gồm ảnh và preparation) trước khi resume. ZIP mặc định không kèm ảnh để tiết kiệm dung lượng tải.


In [ ]:
# Subprocess riêng, không nạp model vào kernel notebook trước.
subprocess.run(command, cwd=CODE_DIR, env=ENV, check=True)


In [ ]:
import json
from IPython.display import FileLink, display

manifest = OUTPUT_ROOT / 'keyframes_run.json'
if manifest.is_file():
    print(json.dumps(json.loads(manifest.read_text()), ensure_ascii=False, indent=2))
for archive in sorted(OUTPUT_ROOT.parent.glob(OUTPUT_ROOT.name + '-from-keyframes*.zip')):
    print(f'{archive.name}: {archive.stat().st_size / 1024**2:.1f} MiB')
    display(FileLink(str(archive)))
print('Artifacts: mapping/, clip-features-32/, objects/, caption_generator/, caption_mapping/, index/, preparation/.')
print('Save Version để giữ output Kaggle trước khi kết thúc session.')
